
# 26 — Modern LLM RL and Preference Optimization

## 01. Critic-Free Advantage Estimation

### Motivation

PPO typically estimates advantages using a learned value
function. Can we avoid training a Critic?

For a fixed prompt, sample G responses independently from
the same policy and score their complete outcomes.

GRPO uses group-normalized rewards:

$$
A_i^{GRPO} =
\frac{R_i-\bar R}{\operatorname{std}(R)+\epsilon}
$$

RLOO uses a leave-one-out baseline:

$$
A_i^{RLOO} =
R_i-\frac{1}{G-1}\sum_{j\ne i}R_j
$$

The two methods require no learned Critic, but their
gradient estimators are not identical.

This section studies only advantage estimation.
PPO-style clipping and token-level optimization come next.


In [1]:
import torch

# (P=2, G=3): three sampled responses for each of two prompts
rewards = torch.tensor([
    [1.0, 0.0, 0.0],  # One correct response
    [0.0, 0.0, 0.0],  # All responses incorrect
])

group_size = rewards.shape[1]

# (P=2, 1): mean and population std within each prompt
group_mean = rewards.mean(dim=1, keepdim=True)
group_std = rewards.std(dim=1, unbiased=False, keepdim=True)

# (P=2, G=3): GRPO group-normalized advantages
grpo_advantages = (rewards - group_mean) / group_std.clamp_min(1e-8)

# (P=2, G=3): average reward of OTHER responses
loo_baselines = (rewards.sum(dim=1, keepdim=True) - rewards) / (group_size - 1)

# (P=2, G=3): RLOO advantages
rloo_advantages = rewards - loo_baselines

print("GRPO advantages:\n", grpo_advantages)
print("RLOO advantages:\n", rloo_advantages)

# No reward contrast -> no group-relative learning signal.
assert torch.all(grpo_advantages[1] == 0)
assert torch.all(rloo_advantages[1] == 0)


GRPO advantages:
 tensor([[ 1.4142, -0.7071, -0.7071],
        [ 0.0000,  0.0000,  0.0000]])
RLOO advantages:
 tensor([[ 1.0000, -0.5000, -0.5000],
        [ 0.0000,  0.0000,  0.0000]])


In [2]:
p = torch.tensor(0.3, dtype=torch.float64)

# (N=4, G=2): all possible pairs of Bernoulli samples
actions = torch.tensor([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=torch.float64)

# (N=4,): exact probabilities of the four sample pairs
pair_probs = torch.where(actions == 1, p, 1 - p).prod(dim=1)

# (N=4, G=2): score-function gradient d log pi(y) / d theta
score_gradients = actions - p

# (N=4, G=2): group-centered advantages
centered = actions - actions.mean(dim=1, keepdim=True)

# (N=4, G=2): leave-one-out advantages
loo = actions - actions.flip(dims=[1])

# (): exact expected policy-gradient estimators
centered_gradient = (pair_probs * (centered * score_gradients).mean(dim=1)).sum()

loo_gradient = (pair_probs * (loo * score_gradients).mean(dim=1)).sum()

# (): exact gradient of E[R] for this Bernoulli policy
true_gradient = p * (1 - p)

print("True gradient:", true_gradient.item())
print("Group-centered:", centered_gradient.item())
print("RLOO:", loo_gradient.item())

torch.testing.assert_close(loo_gradient, true_gradient)
torch.testing.assert_close(centered_gradient, 0.5 * true_gradient)


True gradient: 0.21
Group-centered: 0.105
RLOO: 0.21



### Critic-Free Advantage Estimation — Takeaways

A learned Critic is not the only way to construct a baseline.

Sampling multiple responses to the same prompt allows us
to compare rewards within a group.

GRPO uses group-centered and standardized outcome rewards.

RLOO uses the mean reward of the other samples, excluding
the current response from its own baseline.

Under independent sampling, the RLOO baseline preserves
the expected REINFORCE policy gradient.

Simple group-mean centering instead introduces a fixed
(G-1)/G scaling of the expected gradient.

Standard-deviation normalization changes this analysis.

When all responses receive identical rewards, their
group-relative advantages are zero.

Next: turn group-relative advantages into a token-level
PPO-style objective and derive GRPO.



## 02. From PPO to GRPO: Clipping and Length Bias

GRPO reuses PPO's clipped importance-ratio objective,
but replaces Critic-based advantages with group-relative
outcome advantages.

The original GRPO objective averages token losses separately
within each response:

$$
L_{\mathrm{GRPO}}
=
-\frac1G\sum_{i=1}^{G}
\frac1{T_i}\sum_{t=1}^{T_i}s_{i,t}
$$

This differs from averaging over all valid tokens:

$$
L_{\mathrm{token}}
=
-\frac{\sum_i\sum_t s_{i,t}}{\sum_i T_i}
$$

Dr. GRPO additionally removes group reward standard-deviation
normalization and uses a fixed generation-budget denominator:

$$
L_{\mathrm{Dr.GRPO}}
=
-\frac1{GL_{\max}}
\sum_i\sum_t s_{i,t}^{\mathrm{centered}}
$$

We isolate these differences using two responses with
different lengths.

For simplicity, the KL regularization term is omitted
in this diagnostic. No model training is performed.



In [3]:
from llmfp.rl import ppo_policy_loss

# (G=2,): one correct and one incorrect response
group_rewards = torch.tensor([1.0, 0.0])

# (G=2,): classic GRPO standardized advantages
grpo_adv = (group_rewards - group_rewards.mean()) / group_rewards.std(unbiased=False)

# (G=2,): Dr. GRPO uses unstandardized centered rewards
dr_adv = group_rewards - group_rewards.mean()

# (G=2, T_max=6): first response has 2 tokens, second has 6
action_mask = torch.tensor([
    [True, True, False, False, False, False],
    [True, True, True, True, True, True],
])

lengths = action_mask.sum(dim=1)
group_size, max_length = action_mask.shape

# (G=2, T_max=6): old and current selected-action log-probs
old_logps = torch.full((group_size, max_length), -1.0)
current_logps = old_logps.clone().requires_grad_(True)

# Initially current policy equals old policy: all ratios are 1.
ratios = torch.exp(current_logps - old_logps)
clipped_ratios = ratios.clamp(0.8, 1.2)


def clipped_terms(advantages: torch.Tensor) -> torch.Tensor:
    """Return masked per-token PPO surrogate values."""
    # (G, 1): one advantage broadcast over each response's tokens
    token_adv = advantages[:, None]

    # (G, T_max): PPO clipped surrogate
    surrogate = torch.minimum(ratios * token_adv, clipped_ratios * token_adv)

    return surrogate.masked_fill(~action_mask, 0.0)


# (): original GRPO — average within each response, then across group
grpo_loss = -(clipped_terms(grpo_adv).sum(dim=1) / lengths).mean()

# (): DAPO-style token mean; reuse the Lesson 21 PPO primitive
token_mean_loss, _, _ = ppo_policy_loss(
    current_logps,
    old_logps,
    grpo_adv[:, None].expand_as(current_logps),
    action_mask,
    clip_epsilon=0.2,
)

# (): Dr. GRPO — centered rewards and fixed denominator
dr_grpo_loss = -clipped_terms(dr_adv).sum() / (group_size * max_length)

# (G, T_max): gradients wrt the selected-action log-probabilities
grpo_grad = torch.autograd.grad(grpo_loss, current_logps, retain_graph=True)[0]

token_mean_grad = torch.autograd.grad(
    token_mean_loss, current_logps, retain_graph=True
)[0]

dr_grpo_grad = torch.autograd.grad(dr_grpo_loss, current_logps)[0]

for name, gradient in (
    ("Original GRPO", grpo_grad),
    ("Token mean", token_mean_grad),
    ("Dr. GRPO", dr_grpo_grad),
):
    print(
        f"{name:14s} | "
        f"short token={gradient[0, 0].item():+.4f} | "
        f"long token={gradient[1, 0].item():+.4f}"
    )

# The original GRPO objective gives the short response's
# individual token a larger absolute gradient coefficient.
torch.testing.assert_close(grpo_grad[0, 0].abs(), 3 * grpo_grad[1, 0].abs())

# Fixed-budget normalization removes this inverse-length scaling.
torch.testing.assert_close(dr_grpo_grad[0, 0].abs(), dr_grpo_grad[1, 0].abs())

print("Length-normalization gradient comparison passed.")


Original GRPO  | short token=-0.2500 | long token=+0.0833
Token mean     | short token=-0.1250 | long token=+0.1250
Dr. GRPO       | short token=-0.0417 | long token=+0.0417
Length-normalization gradient comparison passed.



### GRPO Objective and Length Bias — Takeaways

GRPO retains PPO clipping but replaces the learned Critic
with group-relative outcome advantages.

The original GRPO objective averages token losses within
each response before averaging across responses.

This makes the gradient coefficient per token depend on
the individual response length.

Token-mean aggregation and fixed-budget aggregation produce
different gradient weights even with identical rollouts.

Dr. GRPO removes both per-response length normalization
and per-group reward standard-deviation normalization.

Our experiment isolated the gradient coefficients; it did
not train a policy or establish a change in generated lengths.

Next: construct real group rollouts, assign verifiable
rewards, and apply the group-relative PPO objective.



## 03. Training a Tiny Autoregressive Policy with GRPO

We train a two-token policy for one fixed prompt: "Return 11".

The policy has three possible states:

- State 0: before generating the first token.
- State 1: after generating token 0.
- State 2: after generating token 1.

Each state predicts a distribution over the vocabulary {0, 1}.

The only rewarded completion is [1, 1]:

$$
R(y_1,y_2)=\mathbf{1}[(y_1,y_2)=(1,1)]
$$

We sample a group of responses, compute group-normalized
advantages, and update the policy using PPO clipping.

All responses have the same length, so averaging over valid
tokens is equivalent to the original GRPO policy-loss
aggregation in this experiment.

We omit reference KL regularization to isolate the
critic-free group-relative update.


In [4]:
import torch
import torch.nn.functional as F

from llmfp.rl import ppo_policy_loss

# (S=3, V=2): trainable next-token logits for three states
policy_logits = torch.nn.Parameter(torch.zeros(3, 2))


def selected_token_logps(actions: torch.Tensor) -> torch.Tensor:
    """Score sampled two-token responses under the current policy.

    Args:
        actions: (G, T=2) sampled token IDs, each in {0, 1}.

    Returns:
        (G, T=2) conditional log-probabilities.
    """
    # (G,): first-token logP at the start state
    first_logps = F.log_softmax(policy_logits[0], dim=-1)[actions[:, 0]]

    # (G,): second-token logP conditioned on the first token
    second_logps = (
        F
        .log_softmax(policy_logits[1 + actions[:, 0]], dim=-1)
        .gather(1, actions[:, 1:2])
        .squeeze(1)
    )

    return torch.stack([first_logps, second_logps], dim=1)


group_size = 16
generator = torch.Generator().manual_seed(2603)

with torch.no_grad():
    # (G, 1): first sampled token
    first = torch.multinomial(
        torch.softmax(policy_logits[0], dim=-1).expand(group_size, -1),
        num_samples=1,
        generator=generator,
    )

    # (G, 1): second token sampled given the first
    second = torch.multinomial(
        torch.softmax(policy_logits[1 + first[:, 0]], dim=-1),
        num_samples=1,
        generator=generator,
    )

    # (G, T=2): complete sampled responses
    sampled_actions = torch.cat([first, second], dim=1)

    # (G, T=2): frozen behavior-policy log-probabilities
    old_logps = selected_token_logps(sampled_actions).detach().clone()

    # (G,): binary correctness rewards
    rewards = sampled_actions.eq(1).all(dim=1).float()

    # (G,): standardized group-relative advantages
    advantages = (
        (rewards - rewards.mean()) / rewards.std(unbiased=False).clamp_min(1e-8)
    ).detach()

    # (): exact probability of the correct complete response
    probability_before = (
        torch.softmax(policy_logits[0], dim=-1)[1]
        * torch.softmax(policy_logits[2], dim=-1)[1]
    ).item()

print("Sampled responses:", sampled_actions.tolist())
print("Correct responses:", int(rewards.sum().item()), "/", group_size)
print("Exact P(11) before:", probability_before)


Sampled responses: [[0, 0], [1, 1], [1, 1], [0, 1], [1, 0], [1, 0], [0, 0], [0, 0], [0, 1], [1, 1], [1, 1], [0, 1], [1, 0], [0, 0], [1, 1], [1, 1]]
Correct responses: 6 / 16
Exact P(11) before: 0.25


In [5]:
optimizer = torch.optim.SGD([policy_logits], lr=0.5)

# (G=16, T=2): rescore stored actions with differentiable logits
current_logps = selected_token_logps(sampled_actions)

# (G=16, T=2): one outcome advantage shared by both tokens
token_advantages = advantages[:, None].expand_as(current_logps)

# (G=16, T=2): all generated actions are valid
action_mask = torch.ones_like(sampled_actions, dtype=torch.bool)

# (): clipped group-relative policy loss
loss, approx_kl, clip_fraction = ppo_policy_loss(
    current_logps, old_logps, token_advantages, action_mask, clip_epsilon=0.2
)

optimizer.zero_grad(set_to_none=True)
loss.backward()

# (S=3, V=2): gradient of the trainable policy logits
policy_gradients = policy_logits.grad.detach().clone()

optimizer.step()

with torch.no_grad():
    # (): exact probability of the correct complete response
    probability_after = (
        torch.softmax(policy_logits[0], dim=-1)[1]
        * torch.softmax(policy_logits[2], dim=-1)[1]
    ).item()

    # (G=16, T=2): ratios after the update, on stored actions
    updated_ratios = torch.exp(selected_token_logps(sampled_actions) - old_logps)

print("Initial GRPO loss:", loss.item())
print("Policy gradients:\n", policy_gradients)
print("Exact P(11) before:", probability_before)
print("Exact P(11) after:", probability_after)
print("Updated ratio range:", updated_ratios.min().item(), updated_ratios.max().item())

assert 0 < rewards.sum().item() < group_size
assert probability_after > probability_before

print("Critic-free policy improvement verified.")


Initial GRPO loss: -0.0
Policy gradients:
 tensor([[ 0.1694, -0.1694],
        [ 0.0121, -0.0121],
        [ 0.1573, -0.1573]])
Exact P(11) before: 0.25
Exact P(11) after: 0.2924157381057739
Updated ratio range: 0.9154806137084961 1.084519386291504
Critic-free policy improvement verified.



### First GRPO Policy Update — Takeaways

We implemented a tiny autoregressive policy with trainable
next-token logits.

The second-token distribution depends on the first token.

We sampled complete responses from the behavior policy and
recorded their old action log-probabilities.

Verifiable outcome rewards provided a group-relative
learning signal without training a Critic.

The same advantage was assigned to both generated tokens.

We reused PPO's clipped policy loss to perform one
gradient-based update.

The exact probability of the correct completion increased
in our controlled example.

This verifies the basic critic-free policy-update mechanism,
not the effectiveness of full GRPO on real language models.

Next: incorporate reference-policy regularization and examine
how the objective behaves beyond the first update.



## 04. Reference KL: Reward Shaping vs Direct Loss

PPO-RLHF commonly includes a sampled KL penalty in
rollout rewards, then computes fixed advantages.

Original GRPO adds a KL regularizer directly to its loss.

Define:

$$
\delta(a)=\log\pi_\theta(a|s)-\log\pi_{\mathrm{ref}}(a|s)
$$

Two KL estimators are:

$$
k_1(a)=\delta(a)
$$

$$
k_3(a)=e^{-\delta(a)}+\delta(a)-1
$$

Both have expectation equal to forward KL when samples
are drawn from the current policy with appropriate support.

Unlike k1, k3 is nonnegative for individual samples.

With a fixed old-policy rollout, recomputing k3 under the
updated actor gives a differentiable penalty, but its
sample average is not generally an unbiased estimate of
the updated policy's exact KL.


In [6]:
import math

from llmfp.rl import categorical_kl_from_logits

# (S=3, V=2): frozen uniform reference from before Section 03 training
reference_logits = torch.zeros_like(policy_logits).detach()

with torch.no_grad():
    # (S=3, V=2): current and reference log-distributions
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(reference_logits, dim=-1)

    # (S=3, V=2): signed log-probability ratios
    delta = log_p - log_q

    # (S=3, V=2): nonnegative per-action k3 estimator
    k3 = torch.exp(-delta) + delta - 1

    # (S=3,): expectation under the CURRENT policy at each state
    expected_k3 = (log_p.exp() * k3).sum(dim=-1)

    # (S=3,): exact categorical forward KL at each state
    exact_state_kl = categorical_kl_from_logits(policy_logits, reference_logits)

    # (): exact two-token sequence KL via the chain rule
    exact_sequence_kl = exact_state_kl[0] + (log_p[0].exp() * exact_state_kl[1:]).sum()

    # (G=16, T=2): frozen old-policy KL-shaped rewards
    old_reference_logps = torch.full_like(old_logps, -math.log(2))
    frozen_kl_rewards = -0.1 * (old_logps - old_reference_logps)

torch.testing.assert_close(expected_k3, exact_state_kl, atol=5e-7, rtol=1e-4)

torch.testing.assert_close(
    frozen_kl_rewards, torch.zeros_like(frozen_kl_rewards), atol=1e-6, rtol=0
)

print("Exact conditional KL:", exact_state_kl.tolist())
print("Expected k3:", expected_k3.tolist())
print("Exact sequence KL:", exact_sequence_kl.item())
print("Frozen old-rollout KL rewards are zero.")


Exact conditional KL: [0.0035760439932346344, 1.832190901041031e-05, 0.0030849575996398926]
Expected k3: [0.0035760006867349148, 1.8238299162476324e-05, 0.0030849515460431576]
Exact sequence KL: 0.005257278680801392
Frozen old-rollout KL rewards are zero.


In [7]:
kl_beta = 0.1
learning_rate = 0.1

# (G=16, T=2): current logP of previously collected actions
current_action_logps = selected_token_logps(sampled_actions)

# (G=16, T=2): reference logP, uniform for every state
ref_action_logps = torch.full_like(current_action_logps, -math.log(2))

# (G=16, T=2): CURRENT actor minus reference logP
current_delta = current_action_logps - ref_action_logps

# (G=16, T=2): differentiable sampled k3 values
sampled_k3 = torch.exp(-current_delta) + current_delta - 1

# (): PPO clipped policy loss on the frozen rollout
policy_loss, _, _ = ppo_policy_loss(
    current_action_logps, old_logps, token_advantages, action_mask, clip_epsilon=0.2
)

# (): direct sampled KL penalty
kl_penalty = kl_beta * sampled_k3.mean()

# (S=3, V=2): separate gradients for policy and KL terms
policy_grad = torch.autograd.grad(policy_loss, policy_logits, retain_graph=True)[0]

kl_grad = torch.autograd.grad(kl_penalty, policy_logits)[0]

# Two counterfactual optimizer steps from the SAME parameters.
with torch.no_grad():
    # (S=3, V=2): policy-only candidate
    without_kl = policy_logits - learning_rate * policy_grad

    # (S=3, V=2): candidate including the KL gradient
    with_kl = policy_logits - learning_rate * (policy_grad + kl_grad)


def exact_two_token_kl(logits: torch.Tensor) -> torch.Tensor:
    """Return exact full-sequence KL for the two-token policy."""
    # (S=3,): conditional forward KL for each autoregressive state
    state_kl = categorical_kl_from_logits(logits, reference_logits)

    # (V=2,): probability of each first-token outcome
    first_probs = F.softmax(logits[0], dim=-1)

    # (): chain-rule KL over complete two-token responses
    return state_kl[0] + (first_probs * state_kl[1:]).sum()


with torch.no_grad():
    kl_without = exact_two_token_kl(without_kl)
    kl_with = exact_two_token_kl(with_kl)

print("Policy gradient norm:", policy_grad.norm().item())
print("KL gradient norm:", kl_grad.norm().item())
print("Exact KL after policy-only step:", kl_without.item())
print("Exact KL after KL-regularized step:", kl_with.item())

assert kl_grad.norm().item() > 0

print("Direct KL penalty produces an additional gradient.")


Policy gradient norm: 0.3252700865268707
KL gradient norm: 0.0033257973846048117
Exact KL after policy-only step: 0.007578633725643158
Exact KL after KL-regularized step: 0.007551698945462704
Direct KL penalty produces an additional gradient.



### KL Regularization — Takeaways

PPO-RLHF often applies sampled KL shaping during rollout
collection, then optimizes fixed rewards and advantages.

Original GRPO includes a differentiable KL penalty directly
in the policy loss.

The k3 estimator is nonnegative for individual actions.
Its on-policy expectation equals the forward KL.

An unbiased estimate of the KL value does not automatically
provide an unbiased KL gradient when sampled actions are
held fixed.

Reference KL and PPO old-policy ratios serve different roles.

Frozen rollout KL rewards do not change after the actor
updates, while a direct KL penalty can be recomputed.

KL regularization influences the update direction but does
not impose a hard bound on policy divergence.

Next: examine how group sampling, reward variability, and
reference drift interact across repeated updates.



## 05. Uninformative Groups and Sampling Efficiency

For a fixed prompt, suppose each sampled response receives
a binary correctness reward:

$$
R_i \sim \operatorname{Bernoulli}(p)
$$

A group is informative for GRPO's relative reward signal
only when it contains both correct and incorrect responses.

For G independently sampled responses:

$$
P(\text{informative})
=
1-p^G-(1-p)^G
$$

Response diversity is not the same as reward diversity.

Different incorrect responses can all receive reward zero,
giving zero group-relative advantages.

We compare two initial policies on the same two-token task,
without an additional KL penalty. Both use the same GRPO
update rule, but have very different probabilities of
sampling the rewarded completion.


In [8]:
def probability_of_11(logits: torch.Tensor) -> torch.Tensor:
    """Exact probability of the rewarded two-token completion."""
    # State 0 predicts the first token; state 2 predicts
    # the second token conditioned on the first being 1.
    return F.softmax(logits[0], dim=-1)[1] * F.softmax(logits[2], dim=-1)[1]


# (S=3, V=2): uniform initial policy
uniform_start = torch.zeros(3, 2)

# (S=3, V=2): policy making the rewarded completion very rare
rare_start = torch.tensor([[0.0, -3.9], [0.0, 0.0], [0.0, -3.9]])

num_group_samples = 16

for name, logits in (
    ("Uniform", uniform_start),
    ("Rare", rare_start),
):
    # (): exact probability of a rewarded complete response
    p = probability_of_11(logits).item()

    # (): probability that a group has mixed binary rewards
    informative_probability = 1 - p**num_group_samples - (1 - p) ** num_group_samples

    print(f"{name:7s} | P(11)={p:.6f} | P(informative)={informative_probability:.4%}")


Uniform | P(11)=0.250000 | P(informative)=98.9977%
Rare    | P(11)=0.000394 | P(informative)=0.6280%


In [9]:
def sample_binary_responses(
    logits: torch.Tensor,
    n: int,
    rng: torch.Generator,
) -> torch.Tensor:
    """Sample n autoregressive two-token responses."""
    # (G, 1): first sampled token
    first = torch.multinomial(
        F.softmax(logits[0], dim=-1).expand(n, -1),
        1,
        generator=rng,
    )

    # (G, 1): second token conditioned on first
    second = torch.multinomial(
        F.softmax(logits[1 + first[:, 0]], dim=-1),
        1,
        generator=rng,
    )

    return torch.cat([first, second], dim=1)


def score_binary_responses(
    logits: torch.Tensor,
    actions: torch.Tensor,
) -> torch.Tensor:
    """Return (G, 2) selected-action log-probabilities."""
    first_logp = F.log_softmax(logits[0], dim=-1)[actions[:, 0]]

    second_logp = (
        F
        .log_softmax(logits[1 + actions[:, 0]], dim=-1)
        .gather(1, actions[:, 1:2])
        .squeeze(1)
    )

    return torch.stack([first_logp, second_logp], dim=1)


def run_grpo_experiment(
    initial_logits: torch.Tensor,
    seed: int,
    cycles: int = 12,
) -> list[dict]:
    """Train using fresh on-policy groups, without a Critic."""
    current = torch.nn.Parameter(initial_logits.clone())
    optimizer = torch.optim.SGD([current], lr=0.5)
    rng = torch.Generator().manual_seed(seed)

    history = []

    for cycle in range(1, cycles + 1):
        with torch.no_grad():
            # (G=16, T=2): sampled complete responses
            actions = sample_binary_responses(current, num_group_samples, rng)

            # (G,): verifiable binary rewards
            outcomes = actions.eq(1).all(dim=1).float()

            # (G, T=2): behavior-policy log-probs
            old = score_binary_responses(current, actions).detach()

            # (G,): group-relative advantages
            advantages = (outcomes - outcomes.mean()) / outcomes.std(
                unbiased=False
            ).clamp_min(1e-8)

        # (G, T=2): differentiable current action log-probs
        logps = score_binary_responses(current, actions)

        # (): PPO-style GRPO policy loss; equal response lengths
        loss, _, _ = ppo_policy_loss(
            logps,
            old,
            advantages[:, None].expand_as(logps),
            torch.ones_like(actions, dtype=torch.bool),
            clip_epsilon=0.2,
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        gradient_norm = current.grad.norm().item()
        optimizer.step()

        with torch.no_grad():
            p11 = probability_of_11(current).item()

        correct = int(outcomes.sum().item())
        unique = int(torch.unique(actions, dim=0).shape[0])

        # Equal rewards must give a zero policy-gradient signal.
        if correct == 0 or correct == num_group_samples:
            assert gradient_norm == 0.0

        history.append({
            "cycle": cycle,
            "correct": correct,
            "unique": unique,
            "gradient_norm": gradient_norm,
            "p11": p11,
        })

    return history


uniform_history = run_grpo_experiment(uniform_start, seed=2605)
rare_history = run_grpo_experiment(rare_start, seed=2606)

for name, history in (("Uniform", uniform_history), ("Rare", rare_history)):
    informative_groups = sum(0 < row["correct"] < num_group_samples for row in history)

    diverse_groups = sum(row["unique"] > 1 for row in history)

    print(
        f"{name:7s} | "
        f"informative={informative_groups}/{len(history)} | "
        f"diverse={diverse_groups}/{len(history)} | "
        f"final P(11)={history[-1]['p11']:.6f}"
    )


Uniform | informative=12/12 | diverse=12/12 | final P(11)=0.703184
Rare    | informative=0/12 | diverse=12/12 | final P(11)=0.000394



### Zero-Variance Groups — Takeaways

GRPO requires reward variation within a sampled group
to produce group-relative policy-gradient advantages.

Response diversity does not guarantee reward diversity.

For independent binary-reward samples:

$$
P(\text{informative})
=
1-p^G-(1-p)^G
$$

When successful responses are very rare, most sampled
groups may contain only zero rewards.

An optimizer can execute successfully while receiving
zero task-learning gradients.

Repeated on-policy sampling learns from informative groups
but cannot guarantee they will occur.

Dynamic sampling can filter uninformative groups, but
its generation cost may be high for difficult tasks.

A separate KL penalty can still generate an update when
group-relative task advantages are zero; we omitted it
to isolate this effect.

Next: synthesize GRPO, RLOO, Dr. GRPO, and DAPO, and
identify their distinct algorithmic trade-offs.
